# Vi-DFD: chạy hàng đợi run trên Colab

Notebook này chạy lần lượt các config trong `QUEUE` theo đúng `docs/EXPERIMENTS.md`: train, chấm `eval.py`, kiểm tra kết quả, rồi copy thư mục run lên Drive (`<DRIVE_ROOT>/runs/<run_id>/`). Phiên Colab bị ngắt thì mở lại notebook và **Run all**: config nào đã có bản hoàn chỉnh trên Drive sẽ được bỏ qua.

**Chạy song song nhiều tab / nhiều account:** mở notebook này ở nhiều tab Colab (mỗi tab một account Google, mỗi account đều "Add shortcut to Drive" thư mục nhóm vào My Drive với cùng tên), để **cùng một `QUEUE`** và đặt `WORKER_NAME` khác nhau. Trước khi train, tab ghi một file "giữ chỗ" vào `<DRIVE_ROOT>/runs/_claims/`, đợi `CLAIM_SETTLE_SECONDS` cho Drive đồng bộ rồi mới nhận config, nên hai tab không train trùng một config. Trong lúc train, tab cập nhật file đó mỗi 2 phút. Tab chết (hết phiên ~3 giờ) thì sau `STALE_MINUTES` tab khác tự nhận lại config đó và chạy lại từ đầu.

- Không resume run bị ngắt. Run đó được chạy lại từ đầu (lý do ghi ở đầu `scripts/run_queue.py`).
- Phiên T4 miễn phí khoảng 3 giờ: `SESSION_HOURS` làm tab ngừng nhận config mới khi thời gian còn lại không đủ một run, để run không bị cắt giữa chừng.
- Không sửa file nào trong repo. `train.py` từ chối chạy nếu working tree bẩn.
- Trước khi chạy, ghi tên và chọn `Đang chạy` cho các run của mình trong sheet `Vi-DFD_theo_doi_run`.
- Runtime: **GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
# ===== Chỉ sửa ô này =====
import time, uuid
SESSION_START = globals().get("SESSION_START", time.time())   # giữ nguyên khi chạy lại ô này
DRIVE_ROOT = "/content/drive/MyDrive/BTL DL"   # thư mục nhóm; nếu là thư mục được chia sẻ: "Add shortcut to Drive" vào My Drive
TAG = "exp-v1"                                 # đợt 1: exp-v1; đợt 2: exp-v2
RUNNER_REF = "main"                            # nhánh chứa scripts/run_queue.py (PR #1 chưa merge: "feat/tuning-lock-runner")
REPO_URL = "https://github.com/tuan314159265/vi-dfd.git"
PLATFORM = "colab"                             # colab hoặc kaggle
QUEUE = [                                      # Vinh, R07-R18: 12 trial tuning DFD
    "configs/tuning_runs/E2/ti_dfd_*.yaml",
]
# Ví dụ QUEUE khác (ghi đúng file, KHÔNG dùng cả thư mục E3/* hay E5/*):
#   Tuấn đợt 1 (exp-v1):  ["configs/tuning_runs/E2/ti_kd_*.yaml"]
#   Tiến đợt 1 (exp-v1):  ["configs/runs/E2/ti_ce_s4*.yaml", "configs/runs/E2/ti_factorized_s4*.yaml"]
#   Vinh đợt 2 (exp-v2):  ["configs/runs/E2/ti_dfd_s43.yaml", "configs/runs/E2/ti_dfd_s44.yaml",
#                          "configs/runs/E3/no_tone_weight_s4*.yaml", "configs/runs/E3/tone_only_kd_s42.yaml"]
#   Tuấn đợt 2 (exp-v2):  ["configs/runs/E2/ti_kd_s43.yaml", "configs/runs/E2/ti_kd_s44.yaml",
#                          "configs/runs/E3/joint_kd_s4*.yaml", "configs/runs/E3/marginal_joint_kd_s42.yaml"]
#   Tiến đợt 2 (exp-v2):  ["configs/runs/E5/*_fraction*.yaml"]
# Config chỉ để chấm điểm (stage: evaluate, no_validity_mask, temperature_sweep) bị runner bỏ qua.

# --- Chạy song song nhiều tab (mỗi tab một account) ---
WORKER_NAME = "vinh-acc1"                      # ĐỔI cho mỗi tab: vinh-acc1, vinh-acc2, tuan-acc1, ... (chữ, số, gạch ngang)
SHUFFLE = True                                 # mỗi tab đi hàng đợi theo thứ tự riêng, ít khi tranh cùng config
SHARD = None                                   # chia cứng thay vì chỉ dựa vào claim: "0/3", "1/3", "2/3" cho 3 tab
CLAIM_SETTLE_SECONDS = 90                      # đợi Drive đồng bộ claim giữa các account trước khi nhận config
STALE_MINUTES = 15                             # claim không được cập nhật quá lâu = tab đã chết, tab khác nhận lại
RETRY_FAILED = False                           # True: chạy lại cả config đã failed ở tab khác (đọc log trước!)

# --- Thời gian phiên ---
SESSION_HOURS = 3.0                            # Colab T4 miễn phí ~3 giờ; tab không nhận config mới nếu không đủ giờ
RUN_MINUTES = 45                               # ước lượng 1 run (train + eval); tab tự tăng nếu run thật lâu hơn
MAX_RUNS = None                                # ví dụ 4 để dừng sau 4 run trong phiên này

RUN_TESTS = True                               # chạy pytest một lần trước khi train
SMOKE_TEST = False                             # True: thử 1 batch/config, lên runs_smoke/, không bao giờ được tính.
                                               # Lần đầu: SMOKE_TEST=True, MAX_RUNS=1, Run all. Thấy "1 done now ... 0 failed"
                                               # thì đổi SMOKE_TEST=False, MAX_RUNS=None, chạy lại ô tham số và ô "Chạy hàng đợi".
WORKER = f"{WORKER_NAME}-{uuid.uuid4().hex[:4]}"   # thêm mã ngẫu nhiên: lỡ hai tab trùng WORKER_NAME vẫn không đè nhau
print("worker", WORKER)

In [ ]:
import subprocess, sys, os, glob
from pathlib import Path
from google.colab import drive
PY = "/content/py312/bin/python"   # Python 3.12 venv built in section 2 (train.py, eval.py, pytest, runner)
drive.mount("/content/drive")
assert Path(DRIVE_ROOT).is_dir(), f"Không thấy {DRIVE_ROOT}"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0, "Chưa bật GPU: Runtime -> Change runtime type -> GPU"
print(gpu.stdout)

## 1. Code: clone repo, lấy runner, checkout đúng tag

In [ ]:
%%bash -s "$REPO_URL" "$TAG" "$RUNNER_REF"
set -e
cd /content
[ -d vi-dfd ] || git clone -q "$1" vi-dfd
cd vi-dfd
git fetch -q --tags origin
git show "origin/$3:scripts/run_queue.py" > /content/run_queue.py   # runner nằm ngoài repo để working tree sạch
git checkout -q "$2"
git status --porcelain | (! grep .) || { echo "Working tree bẩn"; exit 1; }
echo "HEAD $(git rev-parse --short HEAD) tags: $(git tag --points-at HEAD | tr '
' ' ')"

## 2. Thư viện (theo `docs/EXPERIMENTS.md` mục 3)

In [ ]:
%%bash
set -e
cd /content/vi-dfd
# Colab's own Python may be newer (3.13 has no torchvision 0.20.1 wheel): use Python 3.12 like docs/TEACHER.md
command -v uv >/dev/null || pip install -q uv
[ -x /content/py312/bin/python ] || uv venv -q --python 3.12 /content/py312
uv pip install -q --python /content/py312/bin/python torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
uv pip install -q --python /content/py312/bin/python -r requirements-teacher.txt -r requirements-dev.txt
P=~/.cache/vi-dfd/parseq
[ -d "$P" ] || git clone -q https://github.com/baudm/parseq "$P"
git -C "$P" checkout -q 1902db043c029a7e03a3818c616c06600af574be
/content/py312/bin/python -c "import sys, torch; print('python', sys.version.split()[0]); print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 3. Dữ liệu và teacher: copy từ Drive ra đĩa Colab, kiểm tra SHA-256

In [ ]:
import shutil
DATA = Path("/content/data/vintext")
src = Path(DRIVE_ROOT) / "data"
for part in ("lmdb", "splits"):
    if not (DATA / part).exists():
        print("copy", src / part)
        shutil.copytree(src / part, DATA / part)
found = sorted(glob.glob(f"{DRIVE_ROOT}/teacher/**/best-epoch=17-validation_word_acc=0.8867.ckpt", recursive=True))
assert len(found) == 1, f"Cần đúng 1 checkpoint teacher trong {DRIVE_ROOT}/teacher/, thấy {found}"
TEACHER = Path("/content/teacher") / Path(found[0]).name
if not TEACHER.exists():
    TEACHER.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(found[0], TEACHER)
check = subprocess.run([PY, "scripts/check_data.py", "--data-root", str(DATA), "--teacher-checkpoint", str(TEACHER)],
                       cwd="/content/vi-dfd", capture_output=True, text=True)
print(check.stdout[-1500:], check.stderr[-1500:])
assert check.returncode == 0 and "27/27 files match" in check.stdout, "Dữ liệu hoặc teacher KHÔNG khớp: dừng lại, báo nhóm"

In [ ]:
if RUN_TESTS:
    test = subprocess.run([PY, "-m", "pytest", "-q", "-p", "no:cacheprovider"], cwd="/content/vi-dfd",
                          capture_output=True, text=True)
    print(test.stdout[-2000:])
    assert test.returncode == 0, "pytest không pass: không chạy, báo nhóm"

## 4. Chạy hàng đợi

Bảng tổng kết cuối ô: `busy: claimed by <tab>` là config tab khác đang chạy, `blocked: failed on <tab>` là config đã lỗi ở tab khác (xem log của tab đó; xóa file claim được in ra hoặc đặt `RETRY_FAILED = True` để chạy lại), `not started (time ...)` là không đủ giờ trong phiên này. Mở phiên mới và **Run all** để làm tiếp.

In [ ]:
runs_dir = Path(DRIVE_ROOT) / ("runs_smoke" if SMOKE_TEST else "runs")
budget = SESSION_HOURS * 60 - (time.time() - SESSION_START) / 60   # trừ thời gian cài đặt, copy dữ liệu, pytest
print(f"còn khoảng {budget:.0f} phút cho phiên này")
cmd = [PY, "/content/run_queue.py", *QUEUE, "--repo", "/content/vi-dfd", "--data-root", str(DATA),
       "--teacher-checkpoint", str(TEACHER), "--drive-runs", str(runs_dir), "--require-tag", TAG,
       "--platform", PLATFORM, "--device", "cuda", "--log-dir", "/content/queue_logs",
       "--worker", WORKER, "--claim-settle-seconds", str(CLAIM_SETTLE_SECONDS), "--stale-minutes", str(STALE_MINUTES),
       "--time-budget-minutes", f"{budget:.1f}", "--run-minutes", str(RUN_MINUTES)]
if SHUFFLE:
    cmd.append("--shuffle")
if SHARD:
    cmd += ["--shard", SHARD]
if RETRY_FAILED:
    cmd.append("--retry-failed")
if MAX_RUNS:
    cmd += ["--max-runs", str(MAX_RUNS)]
if SMOKE_TEST:
    cmd.append("--smoke-test")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:   # in từng dòng để thấy tiến độ ngay trong notebook
        print(line, end="")
    exit_code = proc.wait()
except KeyboardInterrupt:      # bấm Stop: dừng train.py và trả claim để tab khác nhận config
    proc.terminate()
    exit_code = proc.wait()
    print("Đã dừng, claim của config đang chạy đã được trả lại")
print("exit code", exit_code, "(0 = không có run failed; config của tab khác hoặc chưa đủ giờ không tính là lỗi)")

## 5. Sau khi chạy

- Mỗi run xong có thư mục `<DRIVE_ROOT>/runs/<run_id>/` với `queue.json` (`status: done`, `worker` = tab đã chạy). Thư mục đuôi `.partial` là bản copy dở: xóa đi.
- `<DRIVE_ROOT>/runs/_claims/` chỉ là file giữ chỗ, không phải kết quả. Đừng xóa claim của tab đang chạy. Claim `failed` thì xóa sau khi đã đọc log, để tab khác chạy lại.
- Hai tab hiếm khi vẫn có thể train trùng một config (Drive đồng bộ chậm hơn `CLAIM_SETTLE_SECONDS`). `e2_tuning.py lock` và `summarize_results.py` sẽ báo trùng: giữ run xong trước, `--exclude` run còn lại.
- Điền sheet: run_id, ngày, GPU, `train_seconds`, word acc validation (in ở bảng tổng kết phía trên), rồi chọn `Xong`.
- Run báo `failed` thì không tự chạy lại nhiều lần, gửi log (`/content/queue_logs/*.log`) vào nhóm.
- Mốc 13, nếu chưa ai đo: `!cd /content/vi-dfd && /content/py312/bin/python scripts/time_kd_step.py --batch-size 64 --train-samples 23267`.